2.Feature Engineering

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

crime_df = spark.table("crime_parquet_table")
crime_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: double (nullable = true)
 |-- Y_Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek, when

crime_df = crime_df.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

crime_df = crime_df \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("IsWeekend", (dayofweek(col("Date")).isin([1,7])).cast("integer"))

crime_df = crime_df.withColumn(
    "label",
    when(col("Arrest") == True, 1).otherwise(0)
)

crime_df.select("Date","Hour","Month","DayOfWeek","IsWeekend","label").show(5)

+-------------------+----+-----+---------+---------+-----+
|               Date|Hour|Month|DayOfWeek|IsWeekend|label|
+-------------------+----+-----+---------+---------+-----+
|2002-12-31 02:32:35|   2|   12|        3|        0|    0|
|2002-12-30 23:30:00|  23|   12|        2|        0|    0|
|2002-12-30 20:24:18|  20|   12|        2|        0|    1|
|2002-12-29 21:00:00|  21|   12|        1|        1|    0|
|2002-12-28 04:30:00|   4|   12|        7|        1|    0|
+-------------------+----+-----+---------+---------+-----+
only showing top 5 rows


In [0]:
selected_columns = [
    "Year",
    "Month",
    "Hour",
    "DayOfWeek",
    "District",
    "Beat",
    "Community_Area",
    "Primary_Type",
    "Domestic",
    "IsWeekend",
    "label"
]

crime_ml_df = crime_df.select(selected_columns).dropna()

crime_ml_df.printSchema()
crime_ml_df.count()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- label: integer (nullable = false)



7888485

In [0]:
from pyspark.sql.functions import col

train_df = crime_ml_df.filter(col("Year") < 2021)
val_df   = crime_ml_df.filter((col("Year") >= 2021) & (col("Year") < 2022))
test_df  = crime_ml_df.filter(col("Year") >= 2022)

print("Train:", train_df.count())
print("Validation:", val_df.count())
print("Test:", test_df.count())

Train: 6653918
Validation: 209613
Test: 1024954
